# 토크나이제이션 이론

## 텍스트가 LLM의 언어가 되는 방식

이 노트북은 토크나이제이션의 개념, 필요성, 그리고 다양한 방식을 상세히 학습합니다.

---

## 1단계: 문제 정의 - 왜 토크나이제이션이 필요한가?

### 1-1. LLM이 이해하는 것

**사람과 LLM의 차이:**

```
사람:
"I love machine learning"
↓ (읽음)
"내가 기계학습을 사랑한다"는 의미를 바로 이해

LLM:
"I love machine learning"
↓ (숫자만 이해 가능)
[318, 1792, 4572, 5231, ...]
↓ (임베딩)
[[0.5, -0.2, 0.8, ...], [0.1, 0.3, -0.5, ...], ...]
↓ (Transformer 처리)
다음 단어 예측
```

**핵심:** LLM은 **숫자(토큰 ID)**만 이해할 수 있다

### 1-2. 텍스트 → 숫자 변환의 과정

**문제:** 어떻게 "machine"이라는 단어를 하나의 숫자로 변환할 것인가?

```
선택지 1: 문자 단위
"machine" → ['m', 'a', 'c', 'h', 'i', 'n', 'e'] → [1, 2, 3, 4, 5, 6, 7]
→ 7개 토큰 (정보 너무 분산됨)

선택지 2: 단어 단위
"machine" → ['machine'] → [4572]
→ 1개 토큰 (하지만 변형 단어는?)

"machines" → ['machines'] → [4573] (새로운 ID 필요)
"machined" → ['machined'] → [4574] (또 새로운 ID)
→ 어휘 폭발 (학습할 단어가 너무 많음)

선택지 3: 토크나이제이션 (적절한 균형)
"machine" → ['machine'] → [4572]
"machines" → ['machine', 's'] → [4572, 100]  (조합으로 표현)
"machined" → ['machine', 'd'] → [4572, 101]
→ 의미와 효율의 균형!
```

### 1-3. 문자 단위 처리의 문제점

**예: "The beautiful sunset"을 문자 단위로 처리**

```
"The beautiful sunset"
↓ (각 문자를 토큰으로)
[4, 8, 6, 1, 2, 6, 1, 18, 20, 9, 6, 21, 12, 1, 19, 21, 14, 19, 6, 20]
↓
20개 토큰!
```

**문제점:**

1. **정보 분산**: 'e'가 여러 번 나오지만 같은 ID → 문맥 손실
2. **시퀀스 길이 증가**: 문장이 너무 길어짐
3. **메모리/계산 폭증**: Attention의 계산량은 O(n²) → 길이가 2배면 계산량 4배
4. **학습 비효율**: 같은 정보를 반복 학습

**실제 영향:**
- 한국어 "기계 학습을 좋아합니다" (11글자)
- 문자 단위: 11개 토큰
- 적절한 토크나이제이션: 5개 토큰
- → 처리 속도 2배 차이, 메모리 2배 차이

### 1-4. 단어 단위 처리의 문제점

**예: 영어의 동사 활용**

```
Base form:    run      (ID: 5000)
Present:      runs     (ID: 5001) - 새로 학습 필요
Past:         ran      (ID: 5002) - 새로 학습 필요
Gerund:       running  (ID: 5003) - 새로 학습 필요
```

**문제점:**

1. **어휘 폭발**: 모든 변형을 개별 단어로 학습
2. **OOV (Out-of-Vocabulary)**: 학습 데이터에 없는 변형은 처리 불가
3. **언어별 복잡성**: 한글은 붙여 쓰므로 단어 경계 불명확

**한글의 예:**

```
"학습을"
- 단어 단위: 하나의 단어? 아니면 "학습" + "을"?
- 띄어쓰기는: "학습을" (붙어있음)
- 의미적으로는: "학습" (명사) + "을" (목적격 조사)
- 단어 단위는 의미를 놓침
```

### 1-5. 토크나이제이션의 해결책

**적절한 크기의 단위 찾기:**

```
문자 단위 (너무 작음)    ←─────────────→    단어 단위 (너무 큼)
                              ↓
                    서브워드 토크나이제이션 ✓
```

**구체적 예:**

```
"beautiful"
↓ (단어 단위 학습시 변형들)
beautiful, beautifully, beautifulness, beautify, ...

↓ (토크나이제이션)
['beautiful']  또는
['beauti', 'ful']  또는
['beauty', 'ful']
→ 변형을 조합으로 표현 가능!
```

**이점:**

| 측면 | 문자 단위 | 토크나이제이션 | 단어 단위 |
|------|----------|--------------|----------|
| 어휘 크기 | 매우 적음 | 적절함 | 매우 많음 |
| 토큰 개수 | 많음 | 중간 | 적음 |
| OOV 문제 | 없음 | 거의 없음 | 많음 |
| 의미 보존 | 낮음 | 높음 | 높음 |
| 계산량 | 많음 | 중간 | 적음 |

---

## 2단계: 솔루션 비교 - 토크나이제이션의 3가지 주요 방식

### 2-1. BPE (Byte Pair Encoding)

**원리:**

```
Step 1: 모든 문자를 개별 토큰으로 시작
"hello" → ['h', 'e', 'l', 'l', 'o']

Step 2: 데이터에서 가장 자주 나오는 문자 쌍을 찾아 하나의 토큰으로 합침
전체 텍스트에서 'l'과 'l'이 가장 자주 인접 → 'll'로 치환
"hello" → ['h', 'e', 'll', 'o']

Step 3: 반복
이제 'e'와 'll'이 자주 인접 → 'ell'로 치환
"hello" → ['h', 'ell', 'o']

Step 4: 원하는 어휘 크기에 도달할 때까지 반복
최종적으로 vocab_size = 50,000개 정도의 토큰 생성
```

**특징:**
- **Greedy**: 항상 가장 자주 나오는 쌍을 우선 선택
- **문자 기반**: 바이트/문자 레벨에서 시작
- **빈도 기반**: 의미와 무관하게 빈도만 고려

**사용 모델:**
- GPT-2, GPT-3, GPT-4
- Llama

**실제 예:**

```python
GPT-2 BPE 토크나이저로 "Hello world"를 처리하면:

"Hello world"
↓
['Hello', 'Ġworld']  (Ġ는 공백 표시)
↓ (ID로 변환)
[15496, 995]
```

### 2-2. WordPiece

**원리:**

```
Step 1: 미리 정의된 단어 목록 (Vocabulary)에서 시작
vocab = ['the', 'cat', 'is', 'running', ...]

Step 2: 입력 텍스트를 단어로 분할 (띄어쓰기 기준)
"playing" → ['play', 'ing']
("playing"이 vocab에 없으므로 분할)

Step 3: 두 번째 및 이후 토큰에 ##를 붙임 ("이전 토큰의 연속" 표시)
'playing' → ['play', '##ing']

Step 4: 미등록 문자는 [UNK] 토큰으로 처리
```

**특징:**
- **어휘 기반**: 미리 정의된 어휘 사전 사용
- **확률 기반**: 발생 확률이 높은 조합 우선
- **##마크**: 서브워드 연결 명시

**사용 모델:**
- BERT
- RoBERTa
- DistilBERT

**실제 예:**

```python
BERT WordPiece 토크나이저로 "playing" 처리:

"playing"
↓
['play', '##ing']  (##는 이전 토큰의 연속)
↓
[2377, 2572]  (vocab에서 찾은 ID)
```

**한글의 경우:**

```
"학습을 합니다"
↓ (띄어쓰기 기준)
['학습을', '합니다']
↓ (vocab 확인)
['학습', '##을', '합니다']  ("학습을"이 없으므로 분할)
```

### 2-3. SentencePiece

**원리:**

```
Step 1: 모든 언어를 바이트 단위로 통일 처리
영어: "hello world"
한글: "안녕하세요"
일본어: "こんにちは"
↓ (모두 바이트 레벨로)

Step 2: BPE와 유사하게 자주 나오는 바이트 시퀀스를 합침
단, 모든 언어에 동일한 알고리즘 적용

Step 3: 공백을 특수 토큰 ▁로 표시
"hello world" → ['▁hello', '▁world']
공백이 아닌 언어도 동일하게 처리 가능
```

**특징:**
- **언어 독립**: 특정 언어에 의존하지 않음
- **바이트 기반**: 가장 기본적인 단위 (바이트)에서 시작
- **공백 명시**: ▁로 공백 표시 (띄어쓰기 정보 보존)

**사용 모델:**
- mBART (다국어)
- mT5 (다국어)
- XLM-RoBERTa (다국어)
- Llama 2 (영어 + 다국어)

**다국어 처리의 이점:**

```
BPE 또는 WordPiece (영어 중심 학습):
"Hello" → 1개 토큰
"안녕하세요" → 5개 토큰 (언어가 다르면 비효율)

SentencePiece (언어 독립):
"Hello" → 1개 토큰
"안녕하세요" → 3개 토큰
"こんにちは" → 3개 토큰
→ 모든 언어 공평한 처리!
```

### 2-4. 3가지 방식 상세 비교

| 특성 | BPE | WordPiece | SentencePiece |
|------|-----|-----------|---------------|
| **시작점** | 바이트/문자 | 단어 | 바이트 |
| **기준** | 빈도 | 확률 | 빈도 |
| **알고리즘** | Greedy (가장 자주) | 확률 최대화 | BPE와 유사 |
| **언어 특화** | 영어 중심 | 영어 중심 | 언어 무관 |
| **어휘 크기** | 50K | 30K | 32K~64K |
| **공백 처리** | Ġ (공백 추가) | [unused] | ▁ (특수 토큰) |
| **한글 처리** | ⭐ (약함) | ⭐⭐ (중간) | ⭐⭐⭐ (최고) |
| **토큰 개수** | 중간 | 적음 | 중간~많음 |
| **처리 속도** | 빠름 | 매우 빠름 | 느림 |
| **의미 보존** | ⭐⭐ | ⭐⭐⭐ | ⭐⭐ |
| **미등록 단어** | ⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ |
| **다국어 지원** | ⭐ (약함) | ⭐⭐ (중간) | ⭐⭐⭐ (우수) |

---

## 3단계: 맥락 파악 - 언어별 특성

### 3-1. 영어의 특성

**언어 특징:**

```
"I love machine learning!"

특징 1: 공백으로 단어가 명확히 구분됨
"I" | "love" | "machine" | "learning"

특징 2: 문법 변화가 단순함
run → runs (단순 s 추가)
run → running (단순 ing 추가)
run → ran (불규칙, 하지만 패턴 적음)

특징 3: 모든 토크나이저가 영어로 학습됨
BPE, WordPiece, SentencePiece 모두 영어 중심
```

**토크나이제이션 결과:**

```
BPE:           ['I', 'Ġlove', 'Ġmachine', 'Ġlearning', '!']     → 5개
WordPiece:     ['I', 'love', 'machine', 'learning', '!']       → 5개
SentencePiece: ['▁I', '▁love', '▁machine', '▁learning', '!']  → 5개
```

**결론:** 영어는 모든 토크나이저에서 **효율적**

### 3-2. 한글의 특성 (가장 복잡)

**언어 특징:**

```
"기계 학습을 좋아합니다"

특징 1: 붙여 쓰기 (Agglutination)
"학습을" = "학습" (명사) + "을" (목적격 조사)
하지만 띄어쓰기로 분리되지 않음!
→ 단어 경계 불명확

특징 2: 복잡한 문법 변화
"사랑" (기본)
"사랑하다" (동사화)
"사랑합니다" (합니다 체)
"사랑했어" (과거형)
"사랑하고 싶어" (결합)
→ 변형이 무한함

특징 3: 자음/모음의 분해 위험
"한" = ㅎ + ㅏ + ㄴ (3개의 독립 자소)
BPE가 이를 개별 토큰으로 분리하면 의미 손실
```

**각 토크나이저의 처리:**

```
BPE 결과:
"기계 학습을 좋아합니다"
→ ['기', '계', '학', '습', '을', '좋', '아', '합', '니', '다']
→ 10개 토큰 (자음/모음 분리로 최악)
→ 의미 완전 손실

WordPiece 결과 (KoBERT):
"기계 학습을 좋아합니다"
→ ['기계', '학습', '##을', '좋', '##아', '##합니다']
→ 6개 토큰 (띄어쓰기 의존, 완벽하지 않음)

SentencePiece 결과:
"기계 학습을 좋아합니다"
→ ['▁기계', '▁학습', '을', '▁좋', '아합니다']
→ 5개 토큰 (의미 보존, 효율적)
```

**구체적 영향:**

```
비용 계산 (OpenAI API 기준):
입력 토큰당 $0.0005

"기계 학습을 좋아합니다" 처리시:
- BPE (10개):       $0.005
- WordPiece (6개):  $0.003
- SentencePiece (5개): $0.0025

→ BPE는 SentencePiece보다 2배 비쌈!
```

### 3-3. 다국어 처리의 공정성

**문제: 언어별 토큰 수 편차**

```
BPE/WordPiece (영어 중심 학습):

영어:   "Hello world" → 2개 토큰
한글:   "안녕하세요" → 5개 토큰
중국어: "你好世界" → 8개 토큰
아랍어: "مرحبا بالعالم" → 7개 토큰

→ 언어별 처리 효율 심각한 편차
→ 한국/중국 사용자는 영어 사용자보다 2~4배 비쌈!
```

**해결책: SentencePiece (언어 독립)**

```
SentencePiece (모든 언어 동등 학습):

영어:   "Hello world" → 2개 토큰
한글:   "안녕하세요" → 3개 토큰
중국어: "你好世界" → 4개 토큰
아랍어: "مرحبا بالعالم" → 4개 토큰

→ 언어별 편차 최소화
→ 공정한 비용 구조
```

**실제 사례:**

```
ChatGPT 출시 초기 (GPT-3.5, BPE 기반):
- 영어 사용자: 1000 토큰 = $0.002
- 한글 사용자: 같은 의미이지만 2000 토큰 = $0.004
→ 불공정

현재 LLaMA 2 (SentencePiece 기반):
- 모든 언어 동등 처리
- 언어별 비용 차이 최소화
```

---

## 요약

### 토크나이제이션이란
텍스트를 LLM이 이해할 수 있는 **적절한 크기의 단위(토큰)**로 분할하는 과정

### 왜 필요한가
1. LLM은 숫자만 이해 가능
2. 문자 단위: 너무 작음 (정보 분산)
3. 단어 단위: 너무 큼 (어휘 폭발, OOV 문제)
4. 토크나이제이션: 적절한 균형

### 3가지 주요 방식
1. **BPE**: 빈도 기반 (GPT 계열, 영어 최적)
2. **WordPiece**: 확률 기반 (BERT 계열, 의미 중심)
3. **SentencePiece**: 언어 독립 (다국어 표준, 한글 최고)

### 언어별 특성
1. **영어**: 모든 방식에서 효율적
2. **한글**: SentencePiece 필수 (BPE 사용 금지!)
3. **다국어**: 언어별 공정성을 위해 SentencePiece 필수